In [ ]:
import time
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import clear_output, display

# URL do Google Sheets (exportado como CSV)
url = "https://docs.google.com/spreadsheets/d/e/2PACX-1vQsdThO_5fB7mVU4BZWuFmxA9TvnJzYfr04RhD1064FMTrbdr2IKuTXePPTfIEwPkRDOV_k09C2r3gk/pub?gid=120810159&single=true&output=csv"

print("Iniciando o painel em tempo real... (Pressione o botão de parar '■' para encerrar)")

try:
    while True:
        # 1. Carrega e trata os dados atualizados
        df = pd.read_csv(url)
        df.columns = df.columns.str.strip()

        # Identifica coluna de tempo
        coluna_data = None
        for col in df.columns:
            if any(k in col.lower() for k in ['data', 'time', 'timestamp', 'horário', 'horario', 'carimbo']):
                coluna_data = col
                break

        if coluna_data:
            df[coluna_data] = pd.to_datetime(df[coluna_data], errors='coerce')
            df = df.sort_values(by=coluna_data)
            eixo_x = df[coluna_data]
        else:
            eixo_x = df.index

        for col in ['Temperatura', 'Umidade']:
            if col in df.columns:
                # Substitui vírgula por ponto caso seja do tipo texto
                if df[col].dtype == object:
                    df[col] = df[col].astype(str).str.replace(',', '.')
                # Converte para float de forma segura (valores inválidos como '55.6.0' viram NaN)
                df[col] = pd.to_numeric(df[col], errors='coerce')

        # 2. Cálculos para KPIs
        temp_atual = df['Temperatura'].dropna().iloc[-1] if 'Temperatura' in df.columns and not df['Temperatura'].dropna().empty else 0
        temp_media = df['Temperatura'].mean() if 'Temperatura' in df.columns else 0
        temp_max = df['Temperatura'].max() if 'Temperatura' in df.columns else 0
        temp_min = df['Temperatura'].min() if 'Temperatura' in df.columns else 0

        umid_atual = df['Umidade'].dropna().iloc[-1] if 'Umidade' in df.columns and not df['Umidade'].dropna().empty else 0
        umid_media = df['Umidade'].mean() if 'Umidade' in df.columns else 0
        umid_max = df['Umidade'].max() if 'Umidade' in df.columns else 0
        umid_min = df['Umidade'].min() if 'Umidade' in df.columns else 0

        # 3. Criação do Dashboard
        fig = make_subplots(
            rows=2, cols=4,
            specs=[
                [{"type": "indicator"}, {"type": "indicator"}, {"type": "indicator"}, {"type": "indicator"}],
                [{"colspan": 4, "type": "xy"}, None, None, None]
            ],
            row_heights=[0.25, 0.75],
            vertical_spacing=0.12,
            subplot_titles=("", "", "", "", "Evolução Temporal: Temperatura (°C) vs Umidade (%)")
        )

        # KPI 1: Temperatura Atual
        fig.add_trace(go.Indicator(
            mode="number+delta",
            value=temp_atual,
            number={'suffix': " °C", 'font': {'size': 36}},
            title={'text': f"<b>Temp. Atual</b><br><span style='font-size:0.8em;color:gray'>Média: {temp_media:.1f}°C</span>"},
            delta={'reference': temp_media, 'relative': False, 'valueformat': '.1f'},
            domain={'row': 0, 'column': 0}
        ), row=1, col=1)

        # KPI 2: Umidade Atual
        fig.add_trace(go.Indicator(
            mode="number+delta",
            value=umid_atual,
            number={'suffix': " %", 'font': {'size': 36}},
            title={'text': f"<b>Umidade Atual</b><br><span style='font-size:0.8em;color:gray'>Média: {umid_media:.1f}%</span>"},
            delta={'reference': umid_media, 'relative': False, 'valueformat': '.1f'},
            domain={'row': 0, 'column': 1}
        ), row=1, col=2)

        # KPI 3: Máx/Mín Temperatura
        fig.add_trace(go.Indicator(
            mode="number",
            value=temp_max,
            number={'suffix': " °C", 'font': {'size': 32, 'color': '#d9534f'}},
            title={'text': f"<b>Temp. Máx / Mín</b><br><span style='font-size:0.8em;color:gray'>Mínima: {temp_min:.1f}°C</span>"},
            domain={'row': 0, 'column': 2}
        ), row=1, col=3)

        # KPI 4: Máx/Mín Umidade
        fig.add_trace(go.Indicator(
            mode="number",
            value=umid_max,
            number={'suffix': " %", 'font': {'size': 32, 'color': '#0275d8'}},
            title={'text': f"<b>Umidade Máx / Mín</b><br><span style='font-size:0.8em;color:gray'>Mínima: {umid_min:.1f}%</span>"},
            domain={'row': 0, 'column': 3}
        ), row=1, col=4)

        # --- GRÁFICO TEMPORAL ---
        fig.add_trace(go.Scatter(
            x=eixo_x,
            y=df['Temperatura'],
            mode='lines+markers',
            name='Temperatura (°C)',
            line=dict(color='#ff5722', width=2),
            hovertemplate="%{x}<br>Temperatura: <b>%{y:.1f} °C</b><extra></extra>"
        ), row=2, col=1)

        fig.add_trace(go.Scatter(
            x=eixo_x,
            y=df['Umidade'],
            mode='lines+markers',
            name='Umidade (%)',
            line=dict(color='#00bcd4', width=2),
            hovertemplate="%{x}<br>Umidade: <b>%{y:.1f} %</b><extra></extra>"
        ), row=2, col=1)

        # Configuração do Layout
        fig.update_layout(
            title_text="<b>Painel de Monitoramento Ambiental (Tempo Real)</b>",
            title_x=0.5,
            template="plotly_white",
            height=750,
            hovermode="x unified",
            legend=dict(
                orientation="h",
                yanchor="bottom",
                y=0.72,
                xanchor="right",
                x=1
            ),
            xaxis2=dict(
                title="Data / Hora",
                rangeslider=dict(visible=True),
                type="date" if coluna_data else "linear"
            ),
            yaxis2=dict(
                title="Valores"
            )
        )

        # Limpa o gráfico anterior e renderiza o novo atualizado
        clear_output(wait=True)
        display(fig)

        # Aguarda 10 segundos antes da próxima atualização
        time.sleep(10)

except KeyboardInterrupt:
    print("\nMonitoramento interrompido pelo usuário.")